In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#Proyecto Integrador Sistemas Inteligentes: Entrenamiento y evaluación de una CNN para detectar
# si un motociclista esta usando casco o no

#Importar las librerías requeridas
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import optimizers
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Activation
from keras.layers import Convolution2D, MaxPooling2D
import cv2
from google.colab.patches import cv2_imshow

In [3]:
#Definir la ruta de las imágenes
entrenamiento = "/content/drive/MyDrive/Deteccion_de_Casco/imagenes/entrenar"
validacion = "/content/drive/MyDrive/Deteccion_de_Casco/imagenes/validar"

#Definir los hiperparámetros de la arquitectura CNN
epocas = 30
altura, anchura = 150, 150
batch_size = 16
pasos = 100

#Definir los hiperparámetros de las capas convolucionales
kernel1 = 32
kernel1_size = (3, 3)
kernel2 = 64
kernel2_size = (3, 3)
size_pooling = (3, 3)

clases = 2

In [4]:
#Generar datos sintéticos (ayuda cuando tenemos pocos datos de entrenamiento)

entrenar = ImageDataGenerator(rescale=1/200,
                              zoom_range=0.30,
                              horizontal_flip=True)

validar = ImageDataGenerator(rescale=1/200)

#Leemos las imágenes
imagenes_entrenamiento = entrenar.flow_from_directory(entrenamiento,
                                                      target_size=(altura, anchura),
                                                      batch_size=batch_size, class_mode="categorical")
imagenes_validacion = validar.flow_from_directory(validacion,
                                                      target_size=(altura, anchura),
                                                      batch_size=batch_size, class_mode="categorical")


Found 1929 images belonging to 2 classes.
Found 480 images belonging to 2 classes.


In [5]:
#Construir arquitectura de la CNN
ModeloCNN = Sequential()

#Determinar las capas convolucionales
#Primera capa
ModeloCNN.add(Convolution2D(kernel1,kernel1_size,
                            padding="same",input_shape=(altura, anchura, 3),activation="relu"))
#Agregar un submuestreo
ModeloCNN.add(MaxPooling2D(pool_size=size_pooling))
#Segunda capa
ModeloCNN.add(Convolution2D(kernel2,kernel2_size,
                            padding="same",input_shape=(altura, anchura, 3),activation="relu"))
ModeloCNN.add(MaxPooling2D(pool_size=size_pooling))
ModeloCNN.add(Flatten()) # =Aplanar las matrices

#Conectar la MLP (Red Neuronal multicapa)
#Primera capa oculta
ModeloCNN.add(Dense(100, activation="relu")) #Lo que se pone en dense lo define la persona puede ser el num q sea
#Segunda capa oculta
ModeloCNN.add(Dense(50, activation="relu"))
ModeloCNN.add(Dropout(0.3)) #apagar cierto numero de neuronas para q no se sobre entrenen
#Capa de salida
ModeloCNN.add(Dense(clases, activation="softmax"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
#Establecer parámetros del entrenamiento
ModeloCNN.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["acc","mse"])

In [7]:
#Entrenar ModeloCNN

ModeloCNN.fit(imagenes_entrenamiento,
              validation_data=imagenes_validacion, epochs=epocas,
              validation_steps=pasos, verbose=1)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 510s 4s/step - acc: 0.4998 - loss: 0.7300 - mse: 0.2609 - val_acc: 0.6583 - val_loss: 0.6265 - val_mse: 0.2193
Epoch 2/30


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


121/121 ━━━━━━━━━━━━━━━━━━━━ 143s 727ms/step - acc: 0.6446 - loss: 0.6395 - mse: 0.2241 - val_acc: 0.6354 - val_loss: 0.6333 - val_mse: 0.2224
Epoch 3/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 88s 726ms/step - acc: 0.6637 - loss: 0.6302 - mse: 0.2188 - val_acc: 0.6917 - val_loss: 0.6559 - val_mse: 0.2206
Epoch 4/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 88s 726ms/step - acc: 0.6896 - loss: 0.5892 - mse: 0.2007 - val_acc: 0.6521 - val_loss: 0.6515 - val_mse: 0.2268
Epoch 5/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 84s 690ms/step - acc: 0.7057 - loss: 0.5586 - mse: 0.1889 - val_acc: 0.7250 - val_loss: 0.6041 - val_mse: 0.1974
Epoch 6/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 146s 725ms/step - acc: 0.7433 - loss: 0.5145 - mse: 0.1712 - val_acc: 0.7125 - val_loss: 0.5912 - val_mse: 0.2002
Epoch 7/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 139s 701ms/step - acc: 0.7381 - loss: 0.5136 - mse: 0.1713 - val_acc: 0.7229 - val_loss: 0.5790 - val_mse: 0.1910
Epoch 8/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 83s 689ms/step - acc: 0.7652 - loss: 0.4960 - mse:

In [8]:
#Guardar el modelo entrenado de la CNN

#Guarda la arquitectura
ModeloCNN.save("/content/drive/MyDrive/Deteccion_de_Casco/modelo/modeloDeteccionCasco.h5")
ModeloCNN.save_weights("/content/drive/MyDrive/Deteccion_de_Casco/modelo/pesosCasco.weights.h5")

In [ ]:
#Proceso de evaluación o reconocimiento de los patrones

import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os.path

#Imagen a clasificar

rutaImagen = "/content/drive/MyDrive/Deteccion_de_Casco/imagenes/probar/prueba2.jpg"
mostrar = cv2.imread(rutaImagen)
cv2_imshow(mostrar)
altura,anchura = 150,150
#cargar el modelo: arquitectura y pesos

modelo = "/content/drive/MyDrive/Deteccion_de_Casco/modelo/modeloDeteccionCasco.h5"
pesos = "/content/drive/MyDrive/Deteccion_de_Casco/modelo/pesosCasco.weights.h5"

ModeloCNN = load_model(modelo)
ModeloCNN.load_weights(pesos)

#Transformar la imagen a clasificar
imagen = load_img(rutaImagen, target_size=(altura,anchura))
imagen = img_to_array(imagen)
imagen = np.expand_dims(imagen,axis=0)

prediccion = ModeloCNN.predict(imagen)
print(prediccion)

max = np.argmax(prediccion)

if (max == 0):
  print("No tiene casco")
elif (max == 1):
  print("Si tiene casco")

In [31]:
!pip install tensorflow --quiet
!pip install tensorflow gradio matplotlib --quiet
!pip install gradio matplotlib --quiet

import os
import gradio as gr
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

In [35]:
# ————————————————————————————
# 1) Parámetros y carga del modelo
# ————————————————————————————
alturaImagen     = 150
anchuraImagen    = 150

# Rutas de tu dataset (solo para extraer clases)
rutaEntrenar     = "/content/drive/MyDrive/Deteccion_de_Casco/imagenes/entrenar"
rutaModelo       = "/content/drive/MyDrive/Deteccion_de_Casco/modelo/modeloDeteccionCasco.h5"
rutaPesos        = "/content/drive/MyDrive/Deteccion_de_Casco/modelo/pesosCasco.weights.h5"

# Carga del modelo y pesos
modeloCNN = load_model(rutaModelo)
modeloCNN.load_weights(rutaPesos)

# ————————————————————————————
# 2) Creación dinámica del mapeo índice→clase
# ————————————————————————————
# Leemos las carpetas en entrenar y las ordenamos alfabéticamente
nombresClases = sorted([
    carpeta for carpeta in os.listdir(rutaEntrenar)
    if os.path.isdir(os.path.join(rutaEntrenar, carpeta))
])
# Creamos el diccionario inverso
idx2clase = {indice: nombre for indice, nombre in enumerate(nombresClases)}

print("Mapeo de clases:", idx2clase)  # Por ejemplo {0:'casco', 1:'nocasco'}

# ————————————————————————————
# 3) Función de detección con lógica dinámica
# ————————————————————————————
def detectarCasco(imagen):
    #Redimensionar
    imagenRedimensionada = cv2.resize(imagen, (anchuraImagen, alturaImagen))
    #Normalizar
    imagenNormalizada   = imagenRedimensionada.astype("float32") / 200.0
    #Preparar batch
    entradaBatch        = np.expand_dims(imagenNormalizada, axis=0)
    #Inferencia
    probabilidades      = modeloCNN.predict(entradaBatch)[0]

    #Índice y nombre crudo de la clase
    indiceMax           = int(np.argmax(probabilidades))
    claseCruda          = idx2clase[indiceMax]  # 'casco' o 'nocasco'

    #Traducir a texto amigable
    if claseCruda.lower() == "casco":
        textoResultado = "Sí tiene casco"
    else:
        textoResultado = "No tiene casco"

    #Gráfico de barras
    etiquetasGrafico = ["Sí tiene casco", "No tiene casco"]
    fig, eje         = plt.subplots(figsize=(4,3))
    colores = ['#2ecc71', '#e74c3c']
    eje.bar(etiquetasGrafico, probabilidades, color=colores)
    eje.set_ylim(0, 1)
    eje.set_ylabel("Probabilidad", color="white")
    eje.set_title("Confianza por clase", color="white")
    eje.tick_params(axis='x', colors="white", rotation=0)
    eje.tick_params(axis='y', colors="white")
    fig.patch.set_alpha(0)
    plt.tight_layout()

    # 3.8 HTML grande para la etiqueta
    htmlEtiqueta = f"<h1 style='text-align:center; color:white; margin:0'>{textoResultado}</h1>"
    return htmlEtiqueta, fig

# ————————————————————————————
# 4) CSS para tamaño uniforme de la imagen
# ————————————————————————————
cssPersonalizado = """
.gradio-container .gr-image img {
    width: 300px !important;
    height: 300px !important;
    object-fit: contain;
    border: 2px solid #ddd;
    border-radius: 8px;
}
"""

# ————————————————————————————
# 5) Construcción de la interfaz
# ————————————————————————————
with gr.Blocks(theme=gr.themes.Soft(), css=cssPersonalizado) as interfaz:
    gr.Markdown("# 🏍️ Detector de Casco para Motociclistas")
    gr.Markdown("Carga tu foto y comprueba si el motociclista lleva casco.")

    with gr.Row():
        with gr.Column():
            entradaImagen  = gr.Image(label="📷 Sube tu foto", type="numpy")
            botonDetectar  = gr.Button("Detectar casco", variant="primary")
        with gr.Column():
            salidaEtiqueta = gr.HTML(label="Resultado")
            salidaGrafico  = gr.Plot(label="Probabilidades")

    botonDetectar.click(
        fn=detectarCasco,
        inputs=entradaImagen,
        outputs=[salidaEtiqueta, salidaGrafico]
    )

    gr.Markdown(
        """
        ---
        **Instrucciones**
        1. Sube una foto clara del motociclista.
        2. Haz clic en **Detectar casco**.
        3. Espera unos segundos y verás el resultado.
        """
    )

# ————————————————————————————
# 6) Lanzar la aplicación
# ————————————————————————————
interfaz.launch(share=True)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Mapeo de clases: {0: 'casco', 1: 'nocasco'}
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77835ca15f449875cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
